# Spectrum reconstruction under growing perturbation: two predictive heads compared

This notebook is the model-facing companion to `part_0_0` and, once built, `part_3`
(latent geometry): the same spectrum, perturbed step by step, reconstructed side by
side by exactly two named models. Two models are compared deliberately, not more — a
third series would make every disagreement figure below ambiguous about which pair is
disagreeing, so `compared` in `analysis_settings.yaml` always names exactly two.

**The `compared` pair below is a placeholder** (`analysis_settings.yaml` names the
`vpu_precision_sweep` condition the user has not yet chosen, and `vpu_precision_sweep`'s own
condition label, not yet confirmed against the real campaign). Which candidate
condition is worth this detailed a look is a judgement best made after `part_4`
(`prediction_global`) ranks the candidates, not before.

## Introduction

The question is whether a difference measured in the latent code (once `part_3` exists)
actually reaches the decoded spectrum, and whether it reaches the classifier's decision
past that. A penalty or objective choice that changes the code without changing the
reconstruction, or changes the reconstruction without changing which molecules are
called, is regularizing or optimizing a quantity that does not propagate.

Three things are tracked as the perturbation grows: the perturbed input, so the size of
the change being asked about is visible; each model's reconstruction, so their
responses can be compared directly; and the disagreement between the two models, drawn
as the band between their reconstructions.

### Assumptions

- **Both models are evaluated with the same physical m/z axis and the same reconstruction
  cost family (`MassersteinLoss`)**, enforced by the shared inference layer
  (`predictive_precompute.prepare_splits`/`precompute`, which requires every ready model
  to declare a Masserstein reconstruction term). Their *parameters* need not be
  identical, unlike the contractive campaign this notebook is ported from, where both
  compared models came from the same sweep; the transport cost below therefore uses one
  fixed, default parameterization for the comparison rather than either model's own
  saved parameters — a deliberate simplification, not an oversight.
- **The two compared models may use different heads.** Unlike the contractive campaign
  (one shared hinge head across the whole sweep), a predictive `compared` pair can name
  a `molecule_binary` condition against a `molecule_pnu` one. `positive_scores` (already
  used by `heads.predictive_comparison.ranking_tables`) converts either head's raw
  output into a comparable positive log-odds score per class, so top-$k$ retention and
  the score-drift measure below are computed on the same footing regardless of head
  type; they are not raw probabilities unless the head itself is a binary sigmoid.
- **Latent variation has not yet been independently checked here.** The contractive
  version of this notebook could cite a specific prior trace measurement before trusting
  a smaller latent response as robustness rather than collapse; no equivalent check
  exists yet for this campaign (`part_3`, latent geometry, is not built). Until it is, a
  smaller angular displacement below should not be read as robustness without also
  checking that the compared model has not simply collapsed its latent variation.
- **Displayed spectra are selected by reconstruction quality, not at random.** Each model
  contributes its two best, two median and two worst clean reconstructions, so the
  figures are a stratified sample of behaviour rather than an arbitrary draw.
- **The zoom window is fixed across amplitudes** within a case, centred on the strongest
  peak of that case's clean spectrum.
- **Both models see identical inputs**, including identical noise draws for the
  stochastic perturbation families at every amplitude, so the comparison is paired at
  the spectrum level.

### Notation

For a spectrum $x$ and transform $T$, write $\delta=T(x)-x$ after TIC renormalization and
$x_t=\mathrm{TIC}(x+t\delta)$ for amplitude $t$. For model $m$ the reconstruction is
$\hat x^m_t=\mathrm{Dec}_m(\mathrm{Enc}_m(x_t))$ and $q_m(x_t)$ the unit latent direction.
Four quantities are reported: the latent displacement $\angle(q_m(x_0),q_m(x_t))$ in
degrees, the reconstruction drift $W(\hat x^m_t,\hat x^m_0)$, the between-model
disagreement $W(\hat x^{A}_t,\hat x^{B}_t)$ (where $W$ is the Masserstein transport cost
on the decoded m/z grid), and the positive log-odds score $s_m(x_t)$ per class, from
which top-$k$ retention and score drift are derived.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

current_path = Path.cwd().resolve()
repository_root = next(path for path in (current_path, *current_path.parents) if (path / "pyproject.toml").is_file())
os.chdir(repository_root)

from msi_autoencoder_wrapper.analysis.autoencoder.experiments import predictive_campaign as campaign
from msi_autoencoder_wrapper.analysis.autoencoder.experiments import predictive_precompute as precompute
from msi_autoencoder_wrapper.analysis.autoencoder.latent import perturbations as perturbation
from msi_autoencoder_wrapper.utils.logger import get_custom_logger
from msi_autoencoder_wrapper.visualization import resolve_theme
from msi_autoencoder_wrapper.visualization.spectra.views import plot_spectrum_comparison

logger = get_custom_logger("reconstruction_local")

from msi_autoencoder_wrapper.analysis.precompute.notebook_inputs import (
    load_model_catalog, load_visualization_theme, select_catalog_frame,
)


## Configuration and the two compared models

## Producing the tables this notebook reads

### Methodology

#### Implementation

No model is loaded in this notebook. Reconstructing the evaluation sample under four
perturbation families at seven amplitudes, for two models with possibly different
heads, is minutes of GPU work, and the figures need the reconstructed spectra
themselves.

That work lives in `predictive_precompute.reconstruction_local`, is configured by
`analysis_settings.yaml` next to this notebook, and writes its tables to this
notebook's results directory. Those spectra are persisted on the decoded m/z grid,
because a spectrum cannot be redrawn from a median.

Run it once, detached, with the command printed below. It refuses to start on the
processor unless explicitly allowed. The command is generated by the runner itself, so
the instruction cannot drift from the entry point it describes.

Everything after this section reads those tables, which is why re-running this
notebook takes seconds.

### Remarks

### Notes

In [ ]:
from msi_autoencoder_wrapper.analysis.precompute.cli import run_precompute_command
SETTINGS_PATH = Path("assets/experiments/autoencoder_architecture/notebooks/segmentation_model/17_09_26_metaspace_heads_vpu_contractive/analysis_settings.yaml")
ANALYSIS = "reconstruction_local"

settings = campaign.load_settings(SETTINGS_PATH)
settings["settings_path"] = str(SETTINGS_PATH)
analysis_settings = precompute.analysis_settings(settings, ANALYSIS)
ARTIFACT_DIR = analysis_settings["output_directory"]

# Seeds and sample settings come from the settings file, so the notebook and the
# precompute cannot drift apart on them.
SAMPLE_SEED = int(settings["sample_seed"])
catalog = load_model_catalog(settings)
LOCAL_ALIASES = list(analysis_settings["selected_models"])
COMPARED = dict(zip(
    catalog.drop_duplicates("model_alias").set_index("model_alias").loc[LOCAL_ALIASES, "display_label"],
    LOCAL_ALIASES,
))
DISPLAY_AMPLITUDES = tuple(analysis_settings["display_amplitudes"])
CURVE_AMPLITUDES = tuple(analysis_settings["curve_amplitudes"])
TOP_K = int(analysis_settings["top_k"])
ZOOM_HALF_WIDTH_MZ = 15.0

theme = resolve_theme(None)
theme.apply()

# --- embedded figure resolution --------------------------------------------
# Inline figures are stored inside the notebook as PNG. Rendering them at 100
# dpi instead of the theme's 140 roughly halves the stored size, leaving figure
# geometry and font proportions untouched.
FIGURE_RASTER_DPI = 100
plt.rcParams["savefig.dpi"] = FIGURE_RASTER_DPI
# ---------------------------------------------------------------------------

print("Run the precompute once, in the background:")
print(f"  {run_precompute_command(SETTINGS_PATH)}")
print("\nOr in the foreground, to watch it:")
print(f"  {run_precompute_command(SETTINGS_PATH, background=False)}")

In [ ]:
# Tables produced by the command above; `load_analysis_table` reports the command again
# if one is missing, so a fresh checkout fails with an instruction rather than a stack
# trace.
from msi_autoencoder_wrapper.analysis.precompute.notebook_inputs import (
    load_model_catalog, load_visualization_theme, select_catalog_frame,
)
models = load_model_catalog(settings)
grid_frame = select_catalog_frame(precompute.load_analysis_table(settings, ANALYSIS, "grid"), models)
case_frame = select_catalog_frame(precompute.load_analysis_table(settings, ANALYSIS, "selected_cases"), models)
panel_frame = select_catalog_frame(precompute.load_analysis_table(settings, ANALYSIS, "case_panels"), models)
spectra_frame = select_catalog_frame(precompute.load_analysis_table(settings, ANALYSIS, "displayed_spectra"), models)
curve_frame = select_catalog_frame(precompute.load_analysis_table(settings, ANALYSIS, "amplitude_curves"), models)
distribution_frame = select_catalog_frame(precompute.load_analysis_table(settings, ANALYSIS, "response_distributions"), models)

provenance = precompute.load_analysis_metadata(settings, ANALYSIS)
HEADS = dict(provenance["compared_heads"])

# The spectra the figures draw, indexed by case row, perturbation and amplitude.
mass_axis = spectra_frame.query("series == 'input' and perturbation == 'clean'")
mass_axis = mass_axis[mass_axis["row"] == mass_axis["row"].iloc[0]]["mz"].to_numpy()
series_lookup = {
    (int(row), str(name), float(amplitude), str(series)): group["intensity"].to_numpy()
    for (row, name, amplitude, series), group in
    spectra_frame.groupby(["row", "perturbation", "amplitude", "series"])
}
shown_rows = provenance["displayed_rows"]
dataset_index_by_row = dict(zip(provenance["displayed_rows"], provenance["displayed_dataset_indices"]))

CATEGORY_ORDER = ("best", "median", "worst")
case_names = {}
for record in sorted(case_frame.to_dict("records"),
                     key=lambda r: (CATEGORY_ORDER.index(r["category"]), r["selected_by"], r["rank"])):
    case_names.setdefault(int(record["row"]), []).append(f"{record['category']} ({record['selected_by']})")
case_label = {row: " / ".join(names) for row, names in case_names.items()}

MODEL_COLOR = {label: theme.color_for_model(label, index) for index, label in enumerate(COMPARED)}
DISAGREEMENT_COLOR = theme.color_for_model("disagreement", 4)

print(f"compared heads: {HEADS}")
print(f"{len(shown_rows)} displayed spectra: {provenance['displayed_dataset_indices']}")
print(f"{len(curve_frame)} amplitude row(s), {len(distribution_frame)} distribution row(s)")
print(f"precomputed at commit {provenance['git_commit']} on {provenance['settings']['device']}")

## Choosing which spectra to display

### Methodology

#### Theoretical

A figure showing one arbitrarily chosen spectrum can mislead in either direction: a
spectrum the models happen to reconstruct well makes any method look competent, and a
pathological one makes both look broken. Selecting by reconstruction quality removes
that arbitrariness and turns the display into a stratified sample of the models'
actual behaviour.

Cases are drawn per model from the two ends and the middle of its own clean-input
Masserstein cost distribution: the two best reconstructions, the two at the median, and
the two worst. Selecting separately for each model matters, because the two need not
find the same spectra easy.

The cost used for selection is the reconstruction cost against the input on clean data,
never a perturbed quantity, so the selection cannot be influenced by the perturbation
response the figures then examine.

#### Implementation

`masserstein_distances` gives the per-spectrum transport cost of each model's clean
reconstruction. Ranks are taken within each model; median cases are the two spectra
adjacent to the median rank. The union of both models' selections is displayed, so a
spectrum chosen by both appears once and carries both labels.

#### Figure descriptions

The table lists every selected case with the model that selected it, its category, its
dataset index, and the clean Masserstein cost under both models, so a case chosen as
one model's worst can be read against how the other model handled it.

### Remarks

### Notes

In [ ]:
# --- what this table shows -------------------------------------------------
CATEGORY = None          # None for every category, or "best" | "median" | "worst"
# ---------------------------------------------------------------------------
shown = case_frame if CATEGORY is None else case_frame.query("category == @CATEGORY")
display(shown.round(4))
print(f"{len(shown_rows)} distinct spectra displayed: {provenance['displayed_dataset_indices']}")

## Reconstruction of the unperturbed spectrum

### Methodology

#### Theoretical

The reference panel. Both models must be seen doing the same job on clean input,
otherwise a difference under perturbation could be a difference in reconstruction
quality rather than in robustness. The band between the two reconstructions on clean
input is the baseline level of the disagreement measure used throughout the rest of
the notebook.

Two views are needed and neither substitutes for the other. The full m/z range shows
where the model spends its capacity across the whole spectrum, including regions the
zoom omits entirely. The zoom around the strongest peak shows the local structure that
the full range compresses into a single vertical line.

#### Implementation

The layout follows the reconstruction analyses elsewhere in this project: over the full
range one signal-and-residual panel pair per model, sharing vertical scales, and for
the zoom a single pair carrying both models together with the disagreement band.
Reconstructions come from the model's `reconstruction` output with no post-processing
beyond the architecture's own TIC output normalization, and every panel is drawn
through the shared `plot_spectrum_comparison`.

#### Figure descriptions

Two figures per selected spectrum, titled with its dataset index, the categories that
selected it and both clean Masserstein costs.

The first covers the complete m/z range with one panel pair per model: the upper panel
of a pair shows intensity against m/z with the input and that model's reconstruction,
the lower shows the signed residual, input minus reconstruction, on a symmetric axis
with a line at zero. Both pairs share their vertical scales, so a taller residual is a
genuinely larger error rather than a rescaled one.

The second is the zoom around the spectrum's strongest peak, with both models on one
panel pair and the area between their reconstructions shaded. Its vertical limit comes
from a high quantile rather than the maximum, since the base peak is an order of
magnitude above the rest; where the peak exceeds the panel its true height is
annotated.

### Remarks

### Notes

In [ ]:
# --- what these figures show -----------------------------------------------
QUANTILE = 0.995          # vertical limit; the base peak is allowed to clip above it
ZOOM_HALF_WIDTH_MZ = 15.0 # half-width of the zoom window around the strongest peak
# ---------------------------------------------------------------------------


def zoom_limits(spectrum: np.ndarray) -> tuple[float, float]:
    """m/z window of fixed width centred on the spectrum's strongest peak."""
    peak_mz = float(mass_axis[int(np.argmax(spectrum))])
    return peak_mz - ZOOM_HALF_WIDTH_MZ, peak_mz + ZOOM_HALF_WIDTH_MZ


def display_limit(traces: list, window: np.ndarray, quantile: float = QUANTILE) -> tuple[float, float]:
    """Vertical limit driven by the bulk of the signal, not by the base peak.

    A TIC-normalized spectrum is dominated by one peak an order of magnitude above the
    rest, so scaling to the maximum flattens every other bin onto the axis.
    """
    values = np.concatenate([np.asarray(trace)[window] for trace in traces])
    limit = float(np.quantile(values, quantile))
    peak = float(values.max())
    if limit <= 0:
        limit = peak if peak > 0 else 1.0
    return 1.15 * limit, peak


def case_costs(row: int, name: str, amplitude: float) -> dict:
    """Masserstein cost of each model's reconstruction, from the precomputed panels."""
    record = panel_frame.query(
        "dataset_index == @dataset_index_by_row[@row] and perturbation == @name and amplitude == @amplitude"
    )
    if not len(record):
        return {label: float("nan") for label in COMPARED}
    return {label: float(record.iloc[0][f"W {label}"]) for label in COMPARED}


for row in shown_rows:
    original = series_lookup[(row, "clean", 0.0, "input")]
    traces = {label: series_lookup[(row, "clean", 0.0, label)] for label in COMPARED}
    costs = {label: float(case_frame.query("row == @row").iloc[0][f"W {label}"]) for label in COMPARED}
    low, high = zoom_limits(original)
    window = (mass_axis >= low) & (mass_axis <= high)

    ## Full range, one signal and residual pair per model, on shared scales
    figure, axes = plt.subplots(
        2 * len(traces), 1, figsize=(14.0, 3.1 * len(traces)), dpi=theme.figure_dpi,
        sharex=True, gridspec_kw={"height_ratios": [2.0, 1.0] * len(traces)},
    )
    signal_max = 1.1 * max([original.max()] + [trace.max() for trace in traces.values()])
    residual_max = 1.1 * max(np.abs(original - trace).max() for trace in traces.values())
    for index, (label, trace) in enumerate(traces.items()):
        signal_axis, residual_axis = axes[2 * index], axes[2 * index + 1]
        plot_spectrum_comparison(mass_axis, original, {label: trace}, axes=(signal_axis, residual_axis), theme=theme)
        signal_axis.set_ylim(0.0, signal_max)
        residual_axis.set_ylim(-residual_max, residual_max)
        signal_axis.text(0.99, 0.92, f"Masserstein = {costs[label]:.4f}", transform=signal_axis.transAxes,
                         ha="right", va="top", fontsize=9)
        if index < len(traces) - 1:
            residual_axis.set_xlabel("")
    axes[0].set_title(
        f"spectrum {dataset_index_by_row[row]} — {case_label.get(row, 'selected')} — "
        "clean, full range (one panel pair per model)", loc="left")
    figure.tight_layout()

    ## Peak neighbourhood, both models with their disagreement band
    figure, (signal_axis, residual_axis) = plot_spectrum_comparison(mass_axis, original, traces, theme=theme)
    figure.set_size_inches(14.0, 7.5)
    signal_axis.fill_between(mass_axis, traces[list(COMPARED)[0]], traces[list(COMPARED)[1]],
                             color=DISAGREEMENT_COLOR, alpha=0.30, linewidth=0, label="between-model disagreement")
    for axis in (signal_axis, residual_axis):
        axis.set_xlim(low, high)
    limit, peak_height = display_limit([original, *traces.values()], window)
    signal_axis.set_ylim(0.0, limit)
    residual_axis.set_ylim(*(1.1 * np.array([-1, 1]) * max(np.abs(original - t)[window].max() for t in traces.values())))
    if peak_height > limit:
        signal_axis.text(0.01, 0.92, f"base peak {peak_height:.3f} clipped", transform=signal_axis.transAxes,
                         fontsize=8, va="top", color="grey")
    signal_axis.legend(fontsize=8, frameon=theme.legend_frame)
    signal_axis.set_title(
        f"spectrum {dataset_index_by_row[row]} — {case_label.get(row, 'selected')} — clean, zoom "
        f"(m/z {0.5 * (low + high):.1f} ± {ZOOM_HALF_WIDTH_MZ:.0f})   |   "
        + ",  ".join(f"{label} W={costs[label]:.4f}" for label in COMPARED), loc="left")
    figure.tight_layout()

## Every perturbation and every amplitude, per case

### Methodology

#### Theoretical

The notebook's central figure, drawn once per selected case. Each row is one
perturbation family and each column one amplitude, so reading a row left to right shows
the spectrum being progressively deformed and both models responding. The leftmost
column is the unperturbed input, so every row starts from the same reference.

The shaded band between the two reconstructions is the quantity of interest. On clean
input it is the models' intrinsic disagreement; as the amplitude grows, any widening is
the two encoders (and, since the two may use different heads, potentially different
objectives entirely) responding differently to the same change. Drawing this for the
best, median and worst reconstructions separately answers a question a single case
cannot: whether an effect is confined to spectra the models already handle well.

#### Implementation

Amplitudes interpolate toward the fully perturbed spectrum and renormalize, so the
direction stays fixed while its size changes. Both models receive identical inputs at
every amplitude, including identical noise draws for the stochastic families. Panels
within a row share their vertical scale, and the m/z window is fixed for the whole
figure, centred on the strongest peak of that case's clean spectrum.

#### Figure descriptions

One figure per selected case, titled with its dataset index and the categories that
selected it. Each is a grid with one pair of rows per perturbation family, named on the
left, and one column per amplitude $t$, named in the panel title together with both
latent angles in degrees. The upper row of a pair shows intensity against m/z with the
perturbed input, both reconstructions, and the shaded disagreement band; the lower
shows the signed residual on a symmetric axis.

### Remarks

### Notes

In [ ]:
# --- what this figure shows ------------------------------------------------
CASES_TO_DRAW = shown_rows      # narrow this to inspect one case at a time
# ---------------------------------------------------------------------------
for case_row in CASES_TO_DRAW:
    clean_series = series_lookup[(case_row, "clean", 0.0, "input")]
    zoom_low, zoom_high = zoom_limits(clean_series)
    zoom_mask = (mass_axis >= zoom_low) & (mass_axis <= zoom_high)

    rows = 2 * len(perturbation.PERTURBATION_NAMES)
    figure, axes = plt.subplots(
        rows, len(DISPLAY_AMPLITUDES),
        figsize=(4.6 * len(DISPLAY_AMPLITUDES), 2.4 * rows), dpi=theme.figure_dpi,
        sharex=True, gridspec_kw={"height_ratios": [2.0, 1.0] * len(perturbation.PERTURBATION_NAMES)},
    )
    for row, name in enumerate(perturbation.PERTURBATION_NAMES):
        row_traces = []
        for amplitude in DISPLAY_AMPLITUDES:
            row_traces.append(series_lookup[(case_row, name, amplitude, "input")])
            row_traces += [series_lookup[(case_row, name, amplitude, label)] for label in COMPARED]
        row_limit, row_peak = display_limit(row_traces, zoom_mask)
        residual_limit = 1.15 * max(
            np.abs(series_lookup[(case_row, name, a, "input")] - series_lookup[(case_row, name, a, label)])[zoom_mask].max()
            for a in DISPLAY_AMPLITUDES for label in COMPARED
        )
        for column, amplitude in enumerate(DISPLAY_AMPLITUDES):
            entry_input = series_lookup[(case_row, name, amplitude, "input")]
            traces = {label: series_lookup[(case_row, name, amplitude, label)] for label in COMPARED}
            signal_axis, residual_axis = axes[2 * row, column], axes[2 * row + 1, column]
            plot_spectrum_comparison(mass_axis, entry_input, traces, axes=(signal_axis, residual_axis), theme=theme)
            signal_axis.fill_between(mass_axis, traces[list(COMPARED)[0]], traces[list(COMPARED)[1]],
                                     color=DISAGREEMENT_COLOR, alpha=0.30, linewidth=0, label="disagreement")
            for axis in (signal_axis, residual_axis):
                axis.set_xlim(zoom_low, zoom_high)
            signal_axis.set_ylim(0.0, row_limit)
            residual_axis.set_ylim(-residual_limit, residual_limit)
            record = panel_frame.query(
                "dataset_index == @dataset_index_by_row[@case_row] and perturbation == @name and amplitude == @amplitude"
            )
            angle_text = ", ".join(f"{label} {float(record.iloc[0][f'angle {label}']):.1f}" for label in COMPARED) if len(record) else ""
            signal_axis.set_title(f"t = {amplitude:g}   |   {angle_text} deg", fontsize=8, loc="left")
            signal_axis.set_ylabel(name if column == 0 else "")
            residual_axis.set_ylabel("input - recon." if column == 0 else "")
            residual_axis.set_xlabel("m/z" if row == len(perturbation.PERTURBATION_NAMES) - 1 else "")
            if row == 0 and column == 0:
                signal_axis.legend(fontsize=7, frameon=theme.legend_frame)
            else:
                for axis in (signal_axis, residual_axis):
                    legend = axis.get_legend()
                    if legend is not None:
                        legend.remove()
        if row_peak > row_limit:
            axes[2 * row, 0].text(0.01, 0.88, f"base peak {row_peak:.3f} clipped",
                                  transform=axes[2 * row, 0].transAxes, fontsize=7, va="top", color="grey")
    figure.suptitle(
        f"spectrum {dataset_index_by_row[case_row]} — {case_label.get(case_row, 'selected')} — "
        f"every perturbation across amplitudes (window m/z {0.5 * (zoom_low + zoom_high):.1f} "
        f"± {ZOOM_HALF_WIDTH_MZ:.0f})", y=1.002, fontsize=13)
    figure.tight_layout()

display(panel_frame.query("amplitude == 1.0").round(4))

## How the models diverge as the perturbation grows

### Methodology

#### Theoretical

The figures above show selected spectra; this section measures the same quantities
across the whole sample so the case-level picture can be checked against the
population.

- **Latent displacement** is the angular change in each model's own canonicalized
  latent direction.
- **Reconstruction drift** is how far each model's output moves from its own clean
  output, measured with the Masserstein cost. Comparing it to the latent displacement
  answers whether a stabilized code reaches the decoder at all.
- **Between-model disagreement** is the Masserstein distance between the two
  reconstructions at the same amplitude. Unlike the first two it needs no per-model
  reference, and its growth with amplitude is the cleanest single statement of how
  differently the two models behave, whether or not they share a head.

Distributions are reported rather than means, because a difference that holds for the
median spectrum and one driven by a tail have different consequences for a decision.

#### Implementation

All three are computed over the full spectrum sample at each amplitude, with identical
inputs and identical noise draws for both models. The zero-amplitude point is evaluated
rather than assumed, as a check that the pipeline introduces no spurious displacement.

#### Figure descriptions

Three rows of panels, one column per perturbation family. The top row plots amplitude
against median latent angular displacement in degrees, the middle against median
reconstruction drift in Masserstein cost, and the bottom against median between-model
disagreement, all on logarithmic axes with the zero-amplitude point excluded and
reported separately. Lines are colored by model in the first two rows; the third has
one line, since disagreement is a property of the pair.

### Remarks

### Notes

In [ ]:
# --- what this figure shows ------------------------------------------------
DRIFT_COLUMN = "median_drift_masserstein"   # or "median_drift_euclidean"
# ---------------------------------------------------------------------------
zero_residual = (
    curve_frame.query("amplitude == 0 and model != 'between models'")[["median_angle_degrees", DRIFT_COLUMN]]
    .abs().max().max()
)
print(f"Zero-amplitude residual (angle and drift): {zero_residual:.3g}")

positive = curve_frame.query("amplitude > 0")
figure, axes = plt.subplots(3, len(perturbation.PERTURBATION_NAMES),
                            figsize=(4.4 * len(perturbation.PERTURBATION_NAMES), 12.0), dpi=theme.figure_dpi)
for column, name in enumerate(perturbation.PERTURBATION_NAMES):
    family = positive.query("perturbation == @name")
    for label in COMPARED:
        run = family.query("model == @label").sort_values("amplitude")
        axes[0, column].plot(run["amplitude"], run["median_angle_degrees"], color=MODEL_COLOR[label],
                             marker="o", markersize=3, linewidth=1.3, label=label)
        axes[1, column].plot(run["amplitude"], run[DRIFT_COLUMN], color=MODEL_COLOR[label],
                             marker="o", markersize=3, linewidth=1.3, label=label)
    pair = family.query("model == 'between models'").sort_values("amplitude")
    axes[2, column].plot(pair["amplitude"], pair[DRIFT_COLUMN], color=DISAGREEMENT_COLOR,
                         marker="o", markersize=3, linewidth=1.4)
    for row, ylabel in ((0, "median latent angle (degrees)"), (1, "median drift, Masserstein"),
                        (2, "median between-model distance")):
        axes[row, column].set_xscale("log")
        axes[row, column].set_yscale("log")
        axes[row, column].set(xlabel="amplitude" if row == 2 else "",
                              ylabel=ylabel if column == 0 else "", title=name if row == 0 else "")
        axes[row, column].grid(theme.grid_visible, alpha=theme.grid_alpha)
axes[0, 0].legend(fontsize=8, frameon=theme.legend_frame)
figure.suptitle("Latent displacement, reconstruction drift and between-model disagreement against amplitude",
                y=1.001, fontsize=13)
figure.tight_layout()

## Spread across spectra at full perturbation

### Methodology

#### Theoretical

The curves above report medians. Whether a median difference reflects the typical
spectrum or is produced by a minority is a separate question, and it decides how far
the result can be relied on. At full perturbation the per-spectrum values are therefore
shown as distributions.

The between-model disagreement is shown at both zero and full amplitude in the same
panel: the clean-input distribution is the models' intrinsic difference, and the
separation between the two distributions is what the perturbation added.

#### Implementation

Every spectrum in the sample contributes one value per family. The violins use all of
them; the overlaid points are a fixed subsample drawn with a seeded generator.

#### Figure descriptions

The left panel shows, per family, the distribution across spectra of the latent angular
displacement at full perturbation, one violin per model, on a logarithmic y-axis. The
middle panel shows reconstruction drift in Masserstein cost in the same layout. The
right panel shows the between-model disagreement with two violins per family, one at
zero amplitude and one at full amplitude, so the widening attributable to the
perturbation is read as the gap between the pair.

### Remarks

### Notes

In [ ]:
# --- what this figure shows ------------------------------------------------
POINT_SAMPLE = 200        # points overlaid per violin; the violin uses every spectrum
AMPLITUDE = 1.0           # amplitude the distributions are taken at
# ---------------------------------------------------------------------------
point_rng = np.random.default_rng(SAMPLE_SEED + 11)


def draw_group(axis, samples_by_position, labels, colors, ylabel, title):
    for position, values in samples_by_position.items():
        values = np.asarray(values)
        values = values[np.isfinite(values) & (values > 0)]
        if not len(values):
            continue
        axis.violinplot([values], positions=[position], widths=0.75, showmedians=True)
        shown = point_rng.choice(values, min(POINT_SAMPLE, len(values)), replace=False)
        axis.scatter(position + point_rng.normal(0.0, 0.06, size=len(shown)), shown,
                     color=colors[position], s=4, alpha=0.3, edgecolor="none")
    axis.set_yscale("log")
    axis.set_xticks(list(labels))
    axis.set_xticklabels(list(labels.values()), rotation=40, ha="right", fontsize=7)
    axis.set(ylabel=ylabel, title=title)
    axis.grid(theme.grid_visible, alpha=theme.grid_alpha)


figure, axes = plt.subplots(1, 3, figsize=(19.0, 5.6), dpi=theme.figure_dpi)
for axis, (column, ylabel, title) in zip(axes[:2], (
    ("angle_degrees", "angular displacement (degrees, log)", f"Latent displacement at t={AMPLITUDE:g}"),
    ("drift_masserstein", "reconstruction drift, Masserstein (log)", f"Reconstruction drift at t={AMPLITUDE:g}"),
)):
    samples, labels, colors, position = {}, {}, {}, 0
    for name in perturbation.PERTURBATION_NAMES:
        for label in COMPARED:
            selected = distribution_frame.query(
                "perturbation == @name and model == @label and amplitude == @AMPLITUDE")
            samples[position] = selected[column].to_numpy()
            labels[position] = f"{name}\n{label}"
            colors[position] = MODEL_COLOR[label]
            position += 1
    draw_group(axis, samples, labels, colors, ylabel, title)

samples, labels, colors, position = {}, {}, {}, 0
for name in perturbation.PERTURBATION_NAMES:
    for amplitude in (0.0, 1.0):
        selected = distribution_frame.query(
            "perturbation == @name and model == 'between models' and amplitude == @amplitude")
        samples[position] = selected["drift_masserstein"].to_numpy()
        labels[position] = f"{name}\nt={amplitude:g}"
        colors[position] = "grey" if amplitude == 0.0 else DISAGREEMENT_COLOR
        position += 1
draw_group(axes[2], samples, labels, colors, "between-model distance, Masserstein (log)",
           "Model disagreement, clean against fully perturbed")
figure.tight_layout()

display(curve_frame.query("amplitude == 1.0")
        .pivot_table(index="perturbation", columns="model", values="median_drift_masserstein").round(5))

## From latent displacement to prediction change

### Methodology

#### Theoretical

Every measurement so far stops at the latent code or the reconstruction. What a
perturbation finally costs is a change in what the model predicts. This section closes
the chain, per pixel, from angular displacement to prediction change.

Two prediction quantities are used, because they answer different questions:

- **Top-$k$ retention** is the fraction of a pixel's $k$ highest-scoring molecules that
  survive the perturbation. It is the decision-level quantity: if a pixel's call set is
  unchanged, the perturbation cost nothing a downstream user would notice, however the
  scores moved underneath.
- **Score drift**, the mean absolute change in the positive log-odds score over
  classes, is the continuous counterpart. It moves even when the ranking does not, so a
  pixel with intact retention but large drift is one whose margins have eroded without
  the decision flipping yet. It is a probability only when the underlying head is a
  binary sigmoid; for a three-state head it is a log-odds difference on the same
  footing, via `positive_scores`, not a literal probability.

#### Implementation

For each perturbation family and amplitude, both models' scores are computed on the
same perturbed spectra used throughout this notebook and compared against their own
clean output per pixel. Retention uses the clean top-$k$ as the reference set. Latent
angles come from the same forward passes as the rest of the notebook, so the pairing
between angle and prediction change is exact at the pixel level.

#### Figure descriptions

Four rows, one per perturbation family, and three columns.

The left column relates the two directly: the x-axis is the pixel's latent angular
displacement in degrees on a logarithmic scale, the y-axis its top-$k$ retention, and
each point is one pixel at full perturbation, colored by model. The displayed case
spectra are overlaid as larger outlined markers. A downward trend means angular
displacement translates into changed predictions; a flat cloud means the code moves
without the decision following.

The middle column shows how retention degrades with amplitude: the x-axis is the
perturbation amplitude, the y-axis the median retention across pixels, one line per
model, with a shaded band spanning the interquartile range.

The right column is the distribution itself at three amplitudes, drawn as step
histograms of per-pixel retention, one color per amplitude and one line style per
model.

### Remarks

### Notes

In [ ]:
# --- what this figure shows ------------------------------------------------
PREDICTION_AMPLITUDES = (0.1, 0.5, 1.0)   # amplitudes drawn in the histogram column
# ---------------------------------------------------------------------------
AMPLITUDE_COLOR = {a: plt.cm.viridis(i / max(len(PREDICTION_AMPLITUDES) - 1, 1))
                   for i, a in enumerate(PREDICTION_AMPLITUDES)}
MODEL_STYLE = dict(zip(COMPARED, ("solid", "dashed")))

figure, axes = plt.subplots(len(perturbation.PERTURBATION_NAMES), 3,
                            figsize=(16.5, 3.7 * len(perturbation.PERTURBATION_NAMES)), dpi=theme.figure_dpi)
for row, name in enumerate(perturbation.PERTURBATION_NAMES):
    scatter_axis, curve_axis, histogram_axis = axes[row]

    for label in COMPARED:
        selected = distribution_frame.query(
            "perturbation == @name and model == @label and amplitude == 1.0")
        scatter_axis.scatter(selected["angle_degrees"], selected["retention"], s=5, alpha=0.16,
                             color=MODEL_COLOR[label], edgecolor="none", label=label)
        highlighted = selected[selected["row"].isin(shown_rows)]
        scatter_axis.scatter(highlighted["angle_degrees"], highlighted["retention"], s=55,
                             facecolors="none", edgecolors=MODEL_COLOR[label], linewidth=1.4)
    scatter_axis.set_xscale("log")
    scatter_axis.set(xlabel="latent angle (degrees, log)", ylabel=f"top-{TOP_K} retention",
                     title=f"{name}: angle against prediction")
    scatter_axis.grid(theme.grid_visible, alpha=theme.grid_alpha)
    if row == 0:
        scatter_axis.legend(fontsize=8, frameon=theme.legend_frame, markerscale=2)

    family = curve_frame.query("perturbation == @name and amplitude > 0")
    for label in COMPARED:
        run = family.query("model == @label").sort_values("amplitude")
        curve_axis.plot(run["amplitude"], run["median_retention"], color=MODEL_COLOR[label],
                        marker="o", markersize=3, linewidth=1.3, label=label)
        curve_axis.fill_between(run["amplitude"], run["q25_retention"], run["q75_retention"],
                                color=MODEL_COLOR[label], alpha=0.15, linewidth=0)
    curve_axis.set_xscale("log")
    curve_axis.set(xlabel="amplitude", ylabel=f"top-{TOP_K} retention", title="median and interquartile range")
    curve_axis.grid(theme.grid_visible, alpha=theme.grid_alpha)

    edges = np.linspace(-0.5 / TOP_K, 1.0 + 0.5 / TOP_K, TOP_K + 2)
    for amplitude in PREDICTION_AMPLITUDES:
        for label in COMPARED:
            values = distribution_frame.query(
                "perturbation == @name and model == @label and amplitude == @amplitude")["retention"]
            if len(values):
                histogram_axis.hist(values, bins=edges, histtype="step", density=True,
                                    color=AMPLITUDE_COLOR[amplitude], linestyle=MODEL_STYLE[label], linewidth=1.3)
    histogram_axis.set(xlabel=f"top-{TOP_K} retention", ylabel="density", title="distribution by amplitude")
    histogram_axis.grid(theme.grid_visible, alpha=theme.grid_alpha)
    if row == 0:
        handles = [plt.Line2D([], [], color=AMPLITUDE_COLOR[a], label=f"t={a:g}") for a in PREDICTION_AMPLITUDES]
        handles += [plt.Line2D([], [], color="grey", linestyle=MODEL_STYLE[label], label=label) for label in COMPARED]
        histogram_axis.legend(handles=handles, fontsize=7, frameon=theme.legend_frame)
figure.suptitle(f"Perturbation to latent angle to prediction: top-{TOP_K} molecule calls per pixel",
                y=1.001, fontsize=13)
figure.tight_layout()

display(curve_frame.query("amplitude == 1.0 and model != 'between models'")
        .pivot_table(index="perturbation", columns="model",
                     values=["median_retention", "unchanged_pixels_fraction", "median_score_drift"]).round(4))

## Reproducible artifacts

### Methodology

#### Implementation

The measured tables and their provenance are written by the precompute runner, not
here. This notebook adds no derived tables of its own beyond a few operations on
tables already in memory. Provenance, including the commit, the device, the seeds and
the source hashes of the analysis modules, lives in the runner's `metadata.json`.

The results directory is deliberately not versioned: it is regenerable from the runner
and its recorded provenance, and the tables are large.

### Remarks

### Notes

In [ ]:
# Every table here is produced by the precompute runner, which also wrote the provenance.
print("precomputed tables present:", sorted(path.name for path in ARTIFACT_DIR.glob("*.csv")))
print("provenance:", {"analysis": provenance["analysis"], "git_commit": provenance["git_commit"],
                      "device": provenance["settings"]["device"], "compared_heads": HEADS,
                      "top_k": provenance["top_k"], "analysed_spectra": provenance["analysed_spectra"]})

## Results / Summary

### LLM

Not yet run against the real campaign cache. `compared` in `analysis_settings.yaml`
still names a placeholder `vpu_precision_sweep` condition; pick the actual condition to
inspect (ideally after `part_4`'s ranking exists), confirm `vpu_precision_sweep`'s real
label, run the precompute command printed above, then write this section from the
actually observed cases, curves and distributions before treating any interpretation
here as verified.

### Person